<a href="https://colab.research.google.com/github/Boni1995/DSE_thesis/blob/main/Booking_Analysis_(cleaned).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
!pip install bertopic
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [ ]:
roberta = SentenceTransformer("all-roberta-large-v1")
topic_model = BERTopic(embedding_model=roberta, verbose=True)

# Positive Reviews

In [ ]:
df_positive = pd.read_csv('/content/df_booking_positive.csv')

In [ ]:
docs_positive = df_positive["review_text"].fillna("").astype(str).tolist()

In [ ]:
topics, probs = topic_model.fit_transform(docs_positive)

2025-08-31 18:24:52,835 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/732 [00:00<?, ?it/s]

2025-08-31 18:28:00,420 - BERTopic - Embedding - Completed ✓
2025-08-31 18:28:00,422 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-31 18:28:52,542 - BERTopic - Dimensionality - Completed ✓
2025-08-31 18:28:52,544 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-31 18:28:57,482 - BERTopic - Cluster - Completed ✓
2025-08-31 18:28:57,491 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-08-31 18:28:58,045 - BERTopic - Representation - Completed ✓


In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

# Negative Reviews

In [ ]:
df_negative = pd.read_csv('/content/df_booking_negative.csv')

In [ ]:
docs_negative = df_negative["review_text"].fillna("").astype(str).tolist()

In [ ]:
topics, probs = topic_model.fit_transform(docs_negative)

2025-08-31 18:28:58,822 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/94 [00:00<?, ?it/s]

2025-08-31 18:29:34,406 - BERTopic - Embedding - Completed ✓
2025-08-31 18:29:34,407 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-31 18:29:59,031 - BERTopic - Dimensionality - Completed ✓
2025-08-31 18:29:59,033 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-31 18:29:59,142 - BERTopic - Cluster - Completed ✓
2025-08-31 18:29:59,147 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-08-31 18:29:59,263 - BERTopic - Representation - Completed ✓


In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

# Trying LDA model

In [ ]:
!pip install gensim

import gensim
import gensim.corpora as corpora
from gensim.models import CoherenceModel

In [ ]:
# Create dictionary and corpus
import ast
df_positive["lemmas"] = df_positive["lemmas"].apply(ast.literal_eval)
id2word = corpora.Dictionary(df_positive["lemmas"])
texts = df_positive["lemmas"]
corpus = [id2word.doc2bow(text) for text in texts]

In [ ]:
# Build LDA model
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                            id2word=id2word,
                                            num_topics=3,
                                            random_state=100,
                                            update_every=1,
                                            chunksize=100,
                                            passes=10,
                                            alpha='auto',
                                            per_word_topics=True)

In [ ]:
# Print the topics
topics = lda_model.print_topics(num_words=10)
for topic in topics:
    print(topic)

(0, '0.044*"location" + 0.036*"good" + 0.034*"staff" + 0.031*"hotel" + 0.030*"breakfast" + 0.022*"room" + 0.022*"nice" + 0.021*"great" + 0.019*"clean" + 0.019*"friendly"')
(1, '0.105*"available" + 0.021*"we" + 0.018*"make" + 0.016*"time" + 0.015*"-" + 0.014*"\'" + 0.013*"like" + 0.013*"day" + 0.012*"check" + 0.011*"leave"')
(2, '0.059*"room" + 0.028*"." + 0.025*"bed" + 0.018*"bathroom" + 0.015*"shower" + 0.013*"floor" + 0.013*"could" + 0.012*"\'" + 0.011*"small" + 0.011*"one"')


In [ ]:
# Compute coherence score
coherence_model_lda = CoherenceModel(model=lda_model, texts=df_positive["lemmas"], dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)


Coherence Score:  0.7526577735543855


After analysing this options, I decided to keep the TF-IDF results as are more clear to show, and provide the same idea of results.